# Conjugate Gradient: Krylov Geometry and Optimal Convergence

The **conjugate gradient** (CG) method is one of the most elegant and practical algorithms in numerical linear algebra. Given a symmetric positive definite (SPD) matrix $A \in \mathbb{R}^{n \times n}$ and a right-hand side $b \in \mathbb{R}^n$, it solves $Ax = b$ by building an $A$-orthogonal basis for successive **Krylov subspaces**:
$$
\mathcal{K}_k(A, r_0) = \mathrm{span}\{r_0,\, Ar_0,\, A^2r_0,\, \ldots,\, A^{k-1}r_0\},
$$
where $r_0 = b - Ax_0$ is the initial residual. The iterate $x_k$ is the unique minimizer of the **$A$-norm error** $\|x_* - x\|_A = \sqrt{(x_* - x)^T A(x_* - x)}$ over the affine subspace $x_0 + \mathcal{K}_k$.

**The algorithm** (Hestenes–Stiefel, 1952) maintains three recurrences:
$$
\alpha_k = \frac{r_k^\top r_k}{p_k^\top A p_k}, \quad
x_{k+1} = x_k + \alpha_k p_k, \quad
r_{k+1} = r_k - \alpha_k A p_k, \quad
\beta_k = \frac{r_{k+1}^\top r_{k+1}}{r_k^\top r_k}, \quad
p_{k+1} = r_{k+1} + \beta_k p_k.
$$

**Convergence** is governed by the **condition number** $\kappa(A) = \lambda_{\max}/\lambda_{\min}$:
$$
\frac{\|e_k\|_A}{\|e_0\|_A} \leq 2\left(\frac{\sqrt{\kappa} - 1}{\sqrt{\kappa} + 1}\right)^k.
$$
For $\kappa = 1$ (identity), CG converges in one step. For large $\kappa$, convergence is slow unless a **preconditioner** is applied.

CG requires only one matrix-vector product per iteration and stores only a handful of vectors — making it the workhorse solver for large sparse SPD systems arising in PDE discretizations, machine learning, and optimization.

## Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatLogSlider, IntSlider

plt.rcParams["figure.dpi"] = 120

## CG and gradient descent implementations

Both methods minimize the quadratic $f(x) = \frac{1}{2}x^\top Ax - b^\top x$, whose unique minimizer is $x_* = A^{-1}b$. **Steepest descent** moves in the direction of $-\nabla f = r_k$ at each step, but the search directions can become nearly parallel (zig-zagging). **CG** enforces that successive search directions are $A$-conjugate: $p_i^\top A p_j = 0$ for $i \neq j$, guaranteeing finite termination in at most $n$ steps.

In [ ]:
def conjugate_gradient(A, b, x0=None, tol=1e-14, max_iter=None):
    """Conjugate Gradient. Returns (iterates, residual_norms)."""
    n = len(b)
    if x0 is None:
        x0 = np.zeros(n)
    if max_iter is None:
        max_iter = n
    x = x0.copy()
    r = b - A @ x
    p = r.copy()
    rr = r @ r
    xs = [x.copy()]
    res = [np.sqrt(rr)]
    for _ in range(max_iter):
        if np.sqrt(rr) < tol:
            break
        Ap = A @ p
        alpha = rr / (p @ Ap)
        x = x + alpha * p
        r = r - alpha * Ap
        rr_new = r @ r
        beta = rr_new / rr
        p = r + beta * p
        rr = rr_new
        xs.append(x.copy())
        res.append(np.sqrt(rr))
    return xs, res


def steepest_descent(A, b, x0=None, tol=1e-14, max_iter=None):
    """Steepest descent (gradient descent on the quadratic). Returns (iterates, residual_norms)."""
    n = len(b)
    if x0 is None:
        x0 = np.zeros(n)
    if max_iter is None:
        max_iter = 5 * n
    x = x0.copy()
    xs = [x.copy()]
    res = [np.linalg.norm(b - A @ x)]
    for _ in range(max_iter):
        r = b - A @ x
        nr = r @ r
        if np.sqrt(nr) < tol:
            break
        alpha = nr / (r @ (A @ r))
        x = x + alpha * r
        xs.append(x.copy())
        res.append(np.linalg.norm(b - A @ x))
    return xs, res


print("Solvers ready.")

## Visualizing the 2D case: contours and search paths

For $n = 2$ we can draw the level sets of $f(x) = \frac{1}{2}x^\top Ax - b^\top x$ as ellipses and overlay the search paths. The aspect ratio of the ellipses reflects the condition number $\kappa$: a large $\kappa$ gives elongated ellipses, causing steepest descent to zig-zag while CG reaches the minimum in exactly 2 steps.

In [ ]:
def plot_2d_paths(kappa=10.0):
    lam1, lam2 = 1.0, kappa
    A2 = np.diag([lam1, lam2])
    x_star = np.array([1.0, 1.0])
    b2 = A2 @ x_star
    x0 = np.array([-1.5, 3.0])

    xs_cg, res_cg   = conjugate_gradient(A2, b2, x0=x0)
    xs_sd, res_sd   = steepest_descent(A2, b2, x0=x0, max_iter=40)

    # level sets
    g = np.linspace(-3, 3, 300)
    G1, G2 = np.meshgrid(g, g)
    Z = 0.5 * (lam1 * G1**2 + lam2 * G2**2) - (b2[0] * G1 + b2[1] * G2)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, (xs, label) in zip(axes,
                                [(xs_sd, "Steepest descent"),
                                 (xs_cg, "Conjugate Gradient")]):
        ax.contour(G1, G2, Z, levels=np.logspace(-1, 2, 20),
                   cmap="Blues", linewidths=0.8, alpha=0.7)
        path = np.array(xs)
        ax.plot(path[:, 0], path[:, 1], "o-", ms=5, lw=1.6,
                color="crimson", label=f"{label} ({len(xs)-1} steps)")
        ax.plot(*x0, "ks", ms=8, label="start")
        ax.plot(*x_star, "g*", ms=12, label="$x^*$")
        ax.set_aspect("equal")
        ax.set_xlim(-2.5, 2.5); ax.set_ylim(-1.5, 3.5)
        ax.set_title(label, fontsize=10)
        ax.legend(fontsize=8)
        ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
    fig.suptitle(f"Quadratic minimization ($n=2$, $\\kappa = {kappa}$)", y=1.02)
    plt.tight_layout()
    plt.show()

plot_2d_paths(kappa=10.0)

## Effect of condition number on convergence

We build diagonal matrices $A = \mathrm{diag}(\lambda_1, \ldots, \lambda_n)$ with eigenvalues spaced to give a prescribed condition number $\kappa$. The theoretical bound
$$
\|e_k\|_A / \|e_0\|_A \leq 2\left(\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\right)^k
$$
is plotted alongside the actual residual curve.

In [ ]:
n = 60
kappas = [2, 10, 100, 1000]
rng = np.random.default_rng(1)
x_true = rng.normal(0, 1, n)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for kappa in kappas:
    lam = np.linspace(1, kappa, n)
    A = np.diag(lam)
    b = A @ x_true
    _, res_cg = conjugate_gradient(A, b, max_iter=n)
    _, res_sd = steepest_descent(A, b, max_iter=5 * n)

    sv = (np.log10(kappa) - np.log10(2)) / (np.log10(1000) - np.log10(2))
    color = (sv, 0, 1 - sv)
    lbl = f"$\\kappa = {kappa}$"

    axes[0].semilogy(np.array(res_cg) / res_cg[0], lw=2, color=color, label=lbl)
    # theoretical bound
    rho = (np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1)
    k_arr = np.arange(len(res_cg))
    axes[0].semilogy(2 * rho**k_arr, lw=1.2, color=color, ls="--", alpha=0.6)

    axes[1].semilogy(np.array(res_sd) / res_sd[0], lw=2, color=color, label=lbl)

axes[0].set_title("Conjugate Gradient  (dashed: theoretical bound)")
axes[1].set_title("Steepest Descent")
for ax in axes:
    ax.set_xlabel("iteration $k$")
    ax.set_ylabel(r"$\|r_k\| / \|r_0\|$")
    ax.legend(fontsize=9)
    ax.set_xlim(0, 120)
fig.suptitle(f"Convergence for $n={n}$ diagonal system, various $\\kappa$", y=1.02)
plt.tight_layout()
plt.show()

## Krylov subspace structure

After $k$ iterations, CG has found the best solution in $x_0 + \mathcal{K}_k(A, r_0)$. For a diagonal $A$ with $d$ **distinct** eigenvalues, CG terminates in exactly $d$ steps regardless of $n$ — a phenomenon called **superlinear convergence** or **clustering**. We illustrate this by using a matrix with only 4 distinct eigenvalues but $n = 40$ dimensions.

In [ ]:
n_clust = 40
# 4 clusters of eigenvalues
lam_clust = np.concatenate([
    np.full(10, 1.0),
    np.full(10, 5.0),
    np.full(10, 20.0),
    np.full(10, 100.0),
])
A_clust = np.diag(lam_clust)
rng2 = np.random.default_rng(2)
x_clust = rng2.normal(0, 1, n_clust)
b_clust = A_clust @ x_clust

# Compare: 4-cluster vs. uniformly spread eigenvalues, same kappa
lam_spread = np.linspace(1, 100, n_clust)
A_spread = np.diag(lam_spread)
b_spread = A_spread @ x_clust

_, res_clust = conjugate_gradient(A_clust, b_clust, max_iter=n_clust)
_, res_spread = conjugate_gradient(A_spread, b_spread, max_iter=n_clust)

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(np.array(res_clust) / res_clust[0], "r-o", ms=4, lw=2,
            label="4 clustered eigenvalues (4 distinct)")
ax.semilogy(np.array(res_spread) / res_spread[0], "b-s", ms=4, lw=2,
            label="uniformly spread eigenvalues (40 distinct)")
ax.axvline(4, color="r", ls="--", lw=1.2, alpha=0.7, label="$k = 4$ clusters")
ax.set_xlabel("iteration $k$")
ax.set_ylabel(r"$\|r_k\|/\|r_0\|$")
ax.set_title(f"CG termination in $d$ steps when $A$ has $d$ distinct eigenvalues ($n={n_clust}$)")
ax.legend()
plt.tight_layout()
plt.show()

## Interactive 2D explorer

Control the condition number and starting point to see how the search paths and iteration counts change.

In [ ]:
def show_2d(log_kappa=1.0):
    plot_2d_paths(kappa=10**log_kappa)

interact(show_2d,
         log_kappa=FloatLogSlider(value=1.0, base=10,
                                   min=0.0, max=3.0, step=0.25,
                                   description=r"$\log_{10}\kappa$"));

## Bibliographical resources

- Hestenes, M. R. and Stiefel, E. (1952). Methods of conjugate gradients for solving linear systems. *Journal of Research of the National Bureau of Standards*, 49(6), 409–436.
- Trefethen, L. N. and Bau, D. (1997). *Numerical Linear Algebra*. SIAM.
- Saad, Y. (2003). *Iterative Methods for Sparse Linear Systems* (2nd ed.). SIAM.
- Nocedal, J. and Wright, S. J. (2006). *Numerical Optimization* (2nd ed.). Springer.
- Shewchuk, J. R. (1994). An introduction to the conjugate gradient method without the agonizing pain. Technical report, Carnegie Mellon University.